In [ ]:
# v6_model — Fix v5 performance regression (0.278 -> 0.1114)
#
# Root causes identified:
#   1. Raw features + PCA components = double-counting (~70 features, ~14 redundant PCA)
#   2. Sentinel -999 fill poisons LightGBM (LightGBM natively handles NaN!)
#   3. swe (Snow Water Equivalent) is all zeros in South Africa = pure noise
#
# v6 fixes:
#   - REMOVED PCA entirely (LightGBM is tree-based, doesn't benefit from PCA)
#   - REMOVED sentinel -999 fill (LightGBM handles NaN natively via optimal split routing)
#   - DROPPED swe (all zeros), duplicate lat/lon columns from merge
#   - REPLACED inf/-inf with NaN in derived features (safe for LightGBM)
#   - ADDED mild regularization: min_child_samples=10
#   - KEPT v3 hyperparameters, multi-seed ensemble, sequential chain, non-negative clipping

import warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, gc, os
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans
from lightgbm import LGBMRegressor
import lightgbm as lgb

# ====================== CONFIG ======================
WORKDIR = Path(".")
SEED = 42
SPATIAL_ROUND = 3
N_SPLITS = 5
N_CLUSTERS = 20
N_SEEDS = 3
SEEDS = [SEED + i * 111 for i in range(N_SEEDS)]  # [42, 153, 264]

TRAIN_LABELS_CSV = WORKDIR / "water_quality_training_dataset.csv"
LANDSAT_TRAIN_CSV = WORKDIR / "landsat_features_training.csv"
LANDSAT_VAL_CSV = WORKDIR / "landsat_features_validation.csv"
TERRACLIMATE_TRAIN_CSV = WORKDIR / "terraclimate_features_training.csv"
TERRACLIMATE_VAL_CSV = WORKDIR / "terraclimate_features_validation.csv"
SUBMISSION_TEMPLATE_CSV = WORKDIR / "submission_template.csv"

OUT_LOCAL = WORKDIR / "submission.csv"
OUT_TMP = "/tmp/submission.csv"

TARGETS = ["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]

# Noise features to drop (swe is all zeros in South Africa)
DROP_FEATURES = ["swe"]

# ====================== SANITY CHECKS ======================
for p in [TRAIN_LABELS_CSV, LANDSAT_TRAIN_CSV, LANDSAT_VAL_CSV,
          TERRACLIMATE_TRAIN_CSV, TERRACLIMATE_VAL_CSV, SUBMISSION_TEMPLATE_CSV]:
    print(p.name, "exists:", p.exists())
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p}")

print("Working directory:", os.getcwd())

# ====================== HELPERS ======================
def parse_date(s):
    return pd.to_datetime(s, format="mixed", dayfirst=True)

def add_month_key_and_round(df, date_col="Sample Date", round_digits=SPATIAL_ROUND):
    df = df.copy()
    df[date_col] = parse_date(df[date_col])
    df["month_ts"] = df[date_col].dt.to_period("M").dt.to_timestamp()
    df["lat_round"] = df["Latitude"].round(round_digits)
    df["lon_round"] = df["Longitude"].round(round_digits)
    return df

# ====================== FEATURE ENGINEERING ======================
def engineer_landsat_features(df):
    """Engineer features from existing Landsat bands."""
    out = df.copy()
    eps = 1e-10

    required = ["nir", "green", "swir16", "swir22"]
    if not all(c in out.columns for c in required):
        print(f"Warning: Missing Landsat columns, skipping engineering")
        return out

    nir = out["nir"].astype(float)
    green = out["green"].astype(float)
    swir16 = out["swir16"].astype(float)
    swir22 = out["swir22"].astype(float)

    # Normalized Difference Indices
    out["NDWI"] = (green - nir) / (green + nir + eps)
    out["NDMI"] = (nir - swir16) / (nir + swir16 + eps)
    out["MNDWI"] = (green - swir16) / (green + swir16 + eps)
    out["MI2"] = (nir - swir22) / (nir + swir22 + eps)
    out["MNDWI2"] = (green - swir22) / (green + swir22 + eps)
    out["SWIR_NDI"] = (swir16 - swir22) / (swir16 + swir22 + eps)

    # Band Ratios
    out["nir_green_ratio"] = nir / (green + eps)
    out["nir_swir16_ratio"] = nir / (swir16 + eps)
    out["nir_swir22_ratio"] = nir / (swir22 + eps)
    out["green_swir16_ratio"] = green / (swir16 + eps)
    out["green_swir22_ratio"] = green / (swir22 + eps)
    out["swir16_swir22_ratio"] = swir16 / (swir22 + eps)

    # Band Statistics
    band_stack = np.column_stack([nir, green, swir16, swir22])
    out["band_mean"] = np.nanmean(band_stack, axis=1)
    out["band_std"] = np.nanstd(band_stack, axis=1)
    out["band_range"] = np.nanmax(band_stack, axis=1) - np.nanmin(band_stack, axis=1)
    out["band_max"] = np.nanmax(band_stack, axis=1)
    out["band_min"] = np.nanmin(band_stack, axis=1)
    out["band_cv"] = out["band_std"] / (out["band_mean"] + eps)

    # Log-transformed bands
    out["log_nir"] = np.log1p(nir.clip(lower=0))
    out["log_green"] = np.log1p(green.clip(lower=0))
    out["log_swir16"] = np.log1p(swir16.clip(lower=0))
    out["log_swir22"] = np.log1p(swir22.clip(lower=0))

    # Interaction terms
    out["nir_x_green"] = nir * green
    out["swir16_x_swir22"] = swir16 * swir22
    out["nir_x_swir16"] = nir * swir16
    out["green_x_swir22"] = green * swir22

    return out


def engineer_terraclimate_features(df):
    """Engineer interaction features from TerraClimate variables."""
    out = df.copy()
    eps = 1e-10

    def get_col(name):
        if name in out.columns: return out[name].astype(float)
        if name + "_terra" in out.columns: return out[name + "_terra"].astype(float)
        return None

    aet = get_col("aet")
    pet = get_col("pet")
    ppt = get_col("ppt")
    defic = get_col("def")
    soil = get_col("soil")
    tmax = get_col("tmax")
    tmin = get_col("tmin")
    srad = get_col("srad")
    vpd = get_col("vpd")
    q = get_col("q")

    available = sum(1 for x in [aet,pet,ppt,defic,soil,tmax,tmin,srad,vpd,q] if x is not None)
    print(f"  TerraClimate variables for engineering: {available}")

    if available < 2:
        print("  Skipping TerraClimate feature engineering (too few variables)")
        return out

    if pet is not None and ppt is not None:
        out["aridity_index"] = pet / (ppt + eps)
    if aet is not None and pet is not None:
        out["aet_pet_ratio"] = aet / (pet + eps)
    if ppt is not None and pet is not None:
        out["water_surplus"] = ppt - pet
    if tmax is not None and tmin is not None:
        out["temp_range"] = tmax - tmin
        out["temp_mean"] = (tmax + tmin) / 2.0
    if q is not None and ppt is not None:
        out["runoff_ratio"] = q / (ppt + eps)
    if soil is not None and defic is not None:
        out["soil_x_deficit"] = soil * defic
    if srad is not None and vpd is not None:
        out["srad_x_vpd"] = srad * vpd

    return out


def add_temporal_features(df, date_col="Sample Date"):
    out = df.copy()
    dt = out[date_col]
    if not pd.api.types.is_datetime64_any_dtype(dt):
        dt = parse_date(dt)
    month = dt.dt.month
    doy = dt.dt.dayofyear
    out["month_sin"] = np.sin(2 * np.pi * month / 12)
    out["month_cos"] = np.cos(2 * np.pi * month / 12)
    out["doy_sin"] = np.sin(2 * np.pi * doy / 365)
    out["doy_cos"] = np.cos(2 * np.pi * doy / 365)
    return out


def add_geo_clusters(train_df, test_df, n_clusters=20, seed=42):
    train_df = train_df.copy()
    test_df  = test_df.copy()
    train_xy = train_df[["Latitude", "Longitude"]].to_numpy(dtype=float)
    test_xy  = test_df[["Latitude", "Longitude"]].to_numpy(dtype=float)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init="auto")
    cluster_labels = kmeans.fit_predict(train_xy)
    centers = kmeans.cluster_centers_
    d2 = ((test_xy[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    test_labels = d2.argmin(axis=1)
    train_df["geo_cluster"] = cluster_labels.astype(int)
    test_df["geo_cluster"]  = test_labels.astype(int)
    return train_df, test_df


# ====================== LOAD DATA ======================
train_labels = pd.read_csv(TRAIN_LABELS_CSV)
landsat_train = pd.read_csv(LANDSAT_TRAIN_CSV)
landsat_val = pd.read_csv(LANDSAT_VAL_CSV)
terraclimate_train = pd.read_csv(TERRACLIMATE_TRAIN_CSV)
terraclimate_val = pd.read_csv(TERRACLIMATE_VAL_CSV)
submission_template = pd.read_csv(SUBMISSION_TEMPLATE_CSV)

print("Loaded shapes:")
print("train_labels:", train_labels.shape)
print("landsat_train:", landsat_train.shape, "landsat_val:", landsat_val.shape)
print("terraclimate_train:", terraclimate_train.shape, "terraclimate_val:", terraclimate_val.shape)

tc_data_cols = [c for c in terraclimate_train.columns if c not in ["Latitude", "Longitude", "Sample Date"]]
print(f"\nTerraClimate data columns ({len(tc_data_cols)}): {tc_data_cols}")

# ====================== PRE-AGGREGATE MONTHLY ======================
landsat_train2 = add_month_key_and_round(landsat_train)
landsat_val2   = add_month_key_and_round(landsat_val)
terr_train2    = add_month_key_and_round(terraclimate_train)
terr_val2      = add_month_key_and_round(terraclimate_val)

landsat_train_m = landsat_train2.groupby(["lat_round","lon_round","month_ts"], as_index=False).mean()
landsat_val_m   = landsat_val2.groupby(["lat_round","lon_round","month_ts"], as_index=False).mean()
terr_train_m    = terr_train2.groupby(["lat_round","lon_round","month_ts"], as_index=False).mean()
terr_val_m      = terr_val2.groupby(["lat_round","lon_round","month_ts"], as_index=False).mean()

print("\nMonthly aggregated shapes:")
print("landsat_train_m:", landsat_train_m.shape, "terr_train_m:", terr_train_m.shape)

# ====================== PREPARE BASE TRAIN/TEST ======================
train = train_labels.copy()
train["Sample Date"] = parse_date(train["Sample Date"])
train["month_ts"] = train["Sample Date"].dt.to_period("M").dt.to_timestamp()
train["lat_round"] = train["Latitude"].round(SPATIAL_ROUND)
train["lon_round"] = train["Longitude"].round(SPATIAL_ROUND)
train["spatial_id"] = train["lat_round"].astype(str) + "_" + train["lon_round"].astype(str)

test = submission_template.copy()
test["Sample Date"] = parse_date(test["Sample Date"])
test["month_ts"] = test["Sample Date"].dt.to_period("M").dt.to_timestamp()
test["lat_round"] = test["Latitude"].round(SPATIAL_ROUND)
test["lon_round"] = test["Longitude"].round(SPATIAL_ROUND)
test["spatial_id"] = test["lat_round"].astype(str) + "_" + test["lon_round"].astype(str)

print("\nPrepared base shapes:", train.shape, test.shape)

# ====================== MERGE ======================
merge_on = ["lat_round", "lon_round", "month_ts"]

train = train.merge(landsat_train_m, how="left", on=merge_on, suffixes=("", "_landsat"))
train = train.merge(terr_train_m,    how="left", on=merge_on, suffixes=("", "_terra"))
test  = test.merge(landsat_val_m,    how="left", on=merge_on, suffixes=("", "_landsat"))
test  = test.merge(terr_val_m,       how="left", on=merge_on, suffixes=("", "_terra"))

print("\nAfter merges: train", train.shape, "test", test.shape)

# Check merge success
for name, df in [("train", train), ("test", test)]:
    landsat_nulls = df["nir"].isna().sum()
    terra_nulls = df["pet"].isna().sum() if "pet" in df.columns else -1
    print(f"  {name}: Landsat NaN={landsat_nulls}/{len(df)} ({landsat_nulls/len(df)*100:.1f}%), "
          f"TerraClimate NaN={terra_nulls}/{len(df)} ({terra_nulls/len(df)*100:.1f}%)")

# ====================== FEATURE ENGINEERING ======================
print("\n--- Feature Engineering ---")
train = engineer_landsat_features(train)
test = engineer_landsat_features(test)
print(f"After Landsat FE: train {train.shape}, test {test.shape}")

print("\n--- TerraClimate Feature Engineering ---")
train = engineer_terraclimate_features(train)
test = engineer_terraclimate_features(test)
print(f"After TerraClimate FE: train {train.shape}, test {test.shape}")

train = add_temporal_features(train)
test = add_temporal_features(test)
print(f"After temporal FE: train {train.shape}, test {test.shape}")

# ====================== CLEAN UP: Replace inf with NaN ======================
# LightGBM handles NaN natively (routes to optimal child at each split)
# NO sentinel -999 fill! That creates artificial patterns.
print("\n--- Cleaning inf values (leaving NaN for LightGBM native handling) ---")
numeric_cols = train.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    train[col] = train[col].replace([np.inf, -np.inf], np.nan)
    test[col] = test[col].replace([np.inf, -np.inf], np.nan)

# ====================== BUILD FEATURE LIST (NO PCA) ======================
exclude = ["Latitude", "Longitude", "Sample Date", "month_ts",
           "lat_round", "lon_round", "spatial_id", "geo_cluster",
           "Latitude_landsat", "Longitude_landsat",
           "Latitude_terra", "Longitude_terra"] + TARGETS

base_features = [c for c in train.columns
                 if c not in exclude
                 and c not in DROP_FEATURES
                 and not c.endswith("_terra")  # drop duplicate suffixed columns from merge
                 and not c.endswith("_landsat")
                 and pd.api.types.is_numeric_dtype(train[c])]

features = list(dict.fromkeys(base_features))

# Also drop any _terra suffixed duplicates that slipped through
features = [f for f in features if f not in DROP_FEATURES]

print(f"\nFeatures count: {len(features)}")
print(f"Features: {features}")

# Diagnostics
nan_counts_train = train[features].isna().sum()
nan_features = nan_counts_train[nan_counts_train > 0]
if len(nan_features) > 0:
    print(f"\nFeatures with NaN (LightGBM handles these natively):")
    for f, cnt in nan_features.items():
        print(f"  {f}: {cnt}/{len(train)} ({cnt/len(train)*100:.1f}%)")
else:
    print("\nNo NaN in features (all merges succeeded)")

# ====================== GEO CLUSTERS ======================
train, test = add_geo_clusters(train, test, n_clusters=N_CLUSTERS, seed=SEED)

n_unique_clusters = train["geo_cluster"].nunique()
print(f"\nGeo clusters: {n_unique_clusters} (N_CLUSTERS={N_CLUSTERS})")
print(f"Cluster sizes: min={train.groupby('geo_cluster').size().min()}, "
      f"max={train.groupby('geo_cluster').size().max()}, "
      f"mean={train.groupby('geo_cluster').size().mean():.0f}")

# ====================== TRAINING ======================
# v3 hyperparameters + mild regularization to prevent overfitting with more features
LGB_PARAMS = dict(
    n_estimators=1200,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=10,   # mild regularization (LightGBM default=20, v3 used default)
    n_jobs=-1,
)

print(f"\nLGB_PARAMS: {LGB_PARAMS}")

oof = {t: np.zeros(len(train)) for t in TARGETS}
models = {t: [] for t in TARGETS}
cv_scores = {}


def train_target_multiseed(target_name, feature_cols):
    y = train[target_name].values.copy()
    X = train[feature_cols]
    groups = train["geo_cluster"].values

    oof_preds_accum = np.zeros(len(train))
    oof_counts = np.zeros(len(train))
    all_seed_models = []
    all_fold_r2 = []

    for seed_idx, seed in enumerate(SEEDS):
        print(f"\n  --- Seed {seed} ({seed_idx+1}/{N_SEEDS}) ---")
        gkf = GroupKFold(n_splits=N_SPLITS)
        seed_models = []

        for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups)):
            Xtr, Xva = X.iloc[tr_idx], X.iloc[va_idx]
            ytr, yva = y[tr_idx], y[va_idx]

            model = LGBMRegressor(**LGB_PARAMS, random_state=seed + fold)
            model.fit(
                Xtr, ytr,
                eval_set=[(Xva, yva)],
                eval_metric="rmse",
                callbacks=[
                    lgb.early_stopping(stopping_rounds=100),
                    lgb.log_evaluation(200),
                ],
            )

            preds_va = model.predict(Xva)
            preds_va = np.clip(preds_va, 0, None)

            oof_preds_accum[va_idx] += preds_va
            oof_counts[va_idx] += 1

            r2 = r2_score(yva, preds_va)
            all_fold_r2.append(r2)
            seed_models.append(model)
            print(f"    Fold {fold} R2: {r2:.4f} (best_iter={model.best_iteration_})")

            del Xtr, Xva, ytr, yva
            gc.collect()

        all_seed_models.append((seed_models, False))

    oof_preds = oof_preds_accum / oof_counts
    oof_r2 = r2_score(y, oof_preds)
    cv_mean = np.mean(all_fold_r2)
    cv_std = np.std(all_fold_r2)

    print(f"\n  >> {target_name}: Pooled OOF R2 = {oof_r2:.4f}")
    print(f"  >> {target_name}: Mean fold R2 = {cv_mean:.4f} +/- {cv_std:.4f}")
    print(f"  >> {target_name}: {N_SEEDS} seeds x {N_SPLITS} folds = {len(all_fold_r2)} models")

    return oof_preds, all_seed_models, (cv_mean, cv_std, oof_r2)


def predict_test_multiseed(seed_models_list, feature_cols, test_df):
    preds = np.zeros(len(test_df))
    n_models = 0
    for seed_models, _ in seed_models_list:
        for model in seed_models:
            preds += model.predict(test_df[feature_cols])
            n_models += 1
    preds /= n_models
    preds = np.clip(preds, 0, None)
    print(f"  Test: averaged {n_models} models, range [{preds.min():.2f}, {preds.max():.2f}]")
    return preds


# ====================== STAGE 1: Electrical Conductance ======================
target_ec = "Electrical Conductance"
print(f"\n{'='*60}")
print(f"=== STAGE 1: {target_ec} ===")
print(f"{'='*60}")

oof_ec, ec_seed_models, ec_cv = train_target_multiseed(target_ec, features)
oof[target_ec] = oof_ec
models[target_ec] = ec_seed_models
cv_scores[target_ec] = ec_cv

test_ec_preds = predict_test_multiseed(ec_seed_models, features, test)
test[target_ec] = test_ec_preds
train["oof_ec"] = oof_ec
test["oof_ec"] = test_ec_preds

# ====================== STAGE 2: Total Alkalinity (uses oof_ec) ======================
target_alk = "Total Alkalinity"
print(f"\n{'='*60}")
print(f"=== STAGE 2: {target_alk} (uses oof_ec) ===")
print(f"{'='*60}")

features_alk = features.copy()
if "oof_ec" not in features_alk:
    features_alk.append("oof_ec")

oof_alk, alk_seed_models, alk_cv = train_target_multiseed(target_alk, features_alk)
oof[target_alk] = oof_alk
models[target_alk] = alk_seed_models
cv_scores[target_alk] = alk_cv

test_alk_preds = predict_test_multiseed(alk_seed_models, features_alk, test)
test[target_alk] = test_alk_preds
train["oof_alk"] = oof_alk
test["oof_alk"] = test_alk_preds

# ====================== STAGE 3: DRP (uses oof_ec + oof_alk) ======================
target_drp = "Dissolved Reactive Phosphorus"
print(f"\n{'='*60}")
print(f"=== STAGE 3: {target_drp} (uses oof_ec + oof_alk) ===")
print(f"{'='*60}")

features_drp = features.copy()
for col in ["oof_ec", "oof_alk"]:
    if col not in features_drp:
        features_drp.append(col)

oof_drp, drp_seed_models, drp_cv = train_target_multiseed(target_drp, features_drp)
oof[target_drp] = oof_drp
models[target_drp] = drp_seed_models
cv_scores[target_drp] = drp_cv

test_drp_preds = predict_test_multiseed(drp_seed_models, features_drp, test)
test[target_drp] = test_drp_preds

# ====================== OOF R2 SUMMARY ======================
print(f"\n{'='*60}")
print("OOF R2 SUMMARY (pooled, original scale):")
print(f"{'='*60}")
oof_r2s = {}
for t in TARGETS:
    r2 = r2_score(train_labels[t].values, oof[t])
    oof_r2s[t] = r2
    print(f"  {t}: {r2:.4f}")

mean_oof_r2 = np.mean(list(oof_r2s.values()))
print(f"\n  MEAN OOF R2 = {mean_oof_r2:.4f}")

# ====================== BUILD SUBMISSION ======================
submission_df = submission_template.copy()
for t in TARGETS:
    submission_df[t] = test[t].values
    submission_df[t] = submission_df[t].clip(lower=0)

submission_df.to_csv(OUT_LOCAL, index=False)
try:
    submission_df.to_csv(OUT_TMP, index=False)
except Exception:
    pass

print(f"\nSaved submission to: {OUT_LOCAL}")
print(f"Also saved to: {OUT_TMP}")

print("\nSubmission preview:")
print(submission_df.describe())

# ====================== SNOWFLAKE UPLOAD ======================
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    sql_cmd = f"PUT file://{OUT_TMP} snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/ AUTO_COMPRESS=FALSE OVERWRITE=TRUE"
    session.sql(sql_cmd).collect()
    print("\nUploaded submission.csv to Snowflake workspace (live).")
except Exception as e:
    print(f"\nSnowflake PUT skipped: {e}")

# ====================== FINAL REPORT ======================
print(f"\n{'='*60}")
print("FINAL CV REPORT")
print(f"{'='*60}")
print(f"N_CLUSTERS={N_CLUSTERS}, N_SPLITS={N_SPLITS}, N_SEEDS={N_SEEDS}")
print(f"Total models per target: {N_SEEDS * N_SPLITS}")
print(f"LightGBM: lr={LGB_PARAMS['learning_rate']}, max_depth={LGB_PARAMS['max_depth']}, "
      f"n_est={LGB_PARAMS['n_estimators']}, min_child_samples={LGB_PARAMS['min_child_samples']}")
print(f"Total features (base): {len(features)}")
print()
for t in TARGETS:
    m, s, oof_r2 = cv_scores[t]
    print(f"  {t}:")
    print(f"    Fold R2 = {m:.4f} +/- {s:.4f}")
    print(f"    OOF R2  = {oof_r2:.4f}")

print(f"\n  MEAN OOF R2 = {mean_oof_r2:.4f}")
print(f"\n  v6 changes vs v5:")
print(f"    - Removed PCA (was adding ~14 redundant features)")
print(f"    - Removed -999 sentinel fill (LightGBM handles NaN natively)")
print(f"    - Dropped swe (all zeros in South Africa)")
print(f"    - Added min_child_samples=10")
print(f"\n  Baselines: v3=0.278, v5=0.1114")